# Install Dependencies And Extract Train/Test Set

In [ ]:
!pip install rarfile --quiet
import rarfile
import os

def extract_rar(file_path, destination):
    os.makedirs(destination, exist_ok=True)
    with rarfile.RarFile(file_path) as rf:
        rf.extractall(destination)
    print(f"Extraction of {file_path} completed.")

# Extract Test
extract_rar('/content/test.rar', '/content/test')

# Extract Train
# extract_rar('/content/train.rar', '/content/train')


Extraction of /content/test.rar completed.


# Train Set Augmentations

In [ ]:
import os
import shutil
import random
from pathlib import Path
from PIL import Image, ImageOps
from google.colab import files

def split_and_augment_single(source_root, aug_name, seed=42):
    """
    source_root: Path to your ORIGINAL (no-augmentation) images.
    aug_name: The specific augmentation to apply (e.g., 'FH')
    """
    random.seed(seed)
    source_root = Path(source_root)

    # Define destination
    base_dest = Path(f"/content/{aug_name}_split_output")
    if base_dest.exists(): shutil.rmtree(base_dest)
    base_dest.mkdir(parents=True)
    print(f"Destination: {base_dest}")

    def get_augmented_image(img, aug_type):
        """Applies the specific transformation based on the aug_type string."""
        fh = ImageOps.mirror(img)
        fv = ImageOps.flip(img)
        r90 = img.rotate(90)

        def translate(base_img, dx, dy):
            return base_img.transform(base_img.size, Image.AFFINE, (1, 0, -dx, 0, 1, -dy), fillcolor=0)

        # Logic for all 12 types
        mapping = {
            "FH": fh,
            "FV": fv,
            "R90_FH": ImageOps.mirror(r90),
            "R90_FV": ImageOps.flip(r90),
            "T_FH_x_ny": translate(fh, 2, -2),
            "T_FH_nx_y": translate(fh, -2, 2),
            "T_FH_nx_ny": translate(fh, -2, -2),
            "T_FH_x_y": translate(fh, 2, 2),
            "T_FV_x_ny": translate(fv, 2, -2),
            "T_FV_nx_y": translate(fv, -2, 2),
            "T_FV_nx_ny": translate(fv, -2, -2),
            "T_FV_x_y": translate(fv, 2, 2),
            "ORIGINAL": img # To get the clean split version
        }
        return mapping.get(aug_type, img)

    print(f"--- Creating {aug_name} Augmentation (Reproducible Split) ---")

    for class_folder in source_root.iterdir():
        if not class_folder.is_dir(): continue
        class_name = class_folder.name
        print(f"Processing Class: {class_name}")

        # 1. Sort and Shuffle Original Data
        images = sorted(list(class_folder.glob("*.png")))
        random.shuffle(images)

        # 2. Split 80/20
        split_idx = int(len(images) * 0.8)
        train_files = images[:split_idx]
        val_files = images[split_idx:]
        print(f"Train: {len(train_files)} | Val: {len(val_files)}")

        # 3. Process and Save
        for dataset_type, file_list in [('train', train_files), ('val', val_files)]:
            target_dir = base_dest / dataset_type / class_name
            target_dir.mkdir(parents=True, exist_ok=True)
            print(f"Processing {dataset_type} for Class: {class_name}")

            for img_path in file_list:
                with Image.open(img_path).convert('L') as img:
                    # Apply the specific augmentation
                    aug_img = get_augmented_image(img, aug_name)

                    # Save with unique name: originalName_augName.png
                    new_name = f"{img_path.stem}_{aug_name}{img_path.suffix}"
                    aug_img.save(target_dir / new_name)
        print(f"Completed Augmentation for Class: {class_name}")

    # 4. Zip and Download
    zip_path = f"/content/{aug_name}_dataset.zip"
    os.system(f"cd {base_dest} && zip -r {zip_path} . > /dev/null")
    files.download(zip_path)

# Example Usage for one type:
# 1. The Base Split (No augmentation)
# split_and_augment_single('/content/original_data', 'ORIGINAL')

# 2. Basic Flips and Rotations
# split_and_augment_single('/content/train', 'FH')
# split_and_augment_single('/content/train', 'FV')
# split_and_augment_single('/content/train', 'R90_FH')
# split_and_augment_single('/content/train', 'R90_FV')

# 3. Horizontal Flip + Translations
# split_and_augment_single('/content/train', 'T_FH_x_ny')
# split_and_augment_single('/content/train', 'T_FH_nx_y')
# split_and_augment_single('/content/train', 'T_FH_nx_ny')
# split_and_augment_single('/content/train', 'T_FH_x_y')

# 4. Vertical Flip + Translations
# split_and_augment_single('/content/train', 'T_FV_x_ny')
# split_and_augment_single('/content/train', 'T_FV_nx_y')
# split_and_augment_single('/content/train', 'T_FV_nx_ny')
# split_and_augment_single('/content/train', 'T_FV_x_y')


Destination: /content/T_FV_nx_y_split_output
--- Creating T_FV_nx_y Augmentation (Reproducible Split) ---
Processing Class: 9
Train: 3200 | Val: 800
Processing train for Class: 9
Processing val for Class: 9
Completed Augmentation for Class: 9
Processing Class: 5
Train: 3200 | Val: 800
Processing train for Class: 5
Processing val for Class: 5
Completed Augmentation for Class: 5
Processing Class: 0
Train: 3200 | Val: 800
Processing train for Class: 0
Processing val for Class: 0
Completed Augmentation for Class: 0
Processing Class: 7
Train: 3200 | Val: 800
Processing train for Class: 7
Processing val for Class: 7
Completed Augmentation for Class: 7
Processing Class: 4
Train: 3200 | Val: 800
Processing train for Class: 4
Processing val for Class: 4
Completed Augmentation for Class: 4
Processing Class: 1
Train: 3200 | Val: 800
Processing train for Class: 1
Processing val for Class: 1
Completed Augmentation for Class: 1
Processing Class: 3
Train: 3200 | Val: 800
Processing train for Class: 3

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Test Set Augmentations

In [ ]:
%%capture
import os
import shutil
from pathlib import Path
from PIL import Image, ImageOps
from google.colab import files
import logging

# 1. Clear any existing logging configurations
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# --- Setup Logging ---
log_file = "/content/test_set_augmentation.log"
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='w' # 'w' overwrites the log each run; use 'a' to append
)


def process_zip_and_cleanup(source_root, split_name):
    source_root = Path(source_root)
    aug_types = [
        "FH", "FV", "R90_FH", "R90_FV",
        "T_FH_x_ny", "T_FH_nx_y", "T_FH_nx_ny", "T_FH_x_y",
        "T_FV_x_ny", "T_FV_nx_y", "T_FV_nx_ny", "T_FV_x_y"
    ]

    base_dest = Path(f"/content/{split_name}_augmentation")
    base_dest.mkdir(exist_ok=True)
    logging.info(f"Augmentation base path: {base_dest}")


    logging.info(f"--- Processing {split_name} Set ---")

    # 1. Image Processing
    for class_folder in source_root.iterdir():
        if not class_folder.is_dir(): continue
        class_name = class_folder.name
        logging.info(f"Processing class: {class_name}")

        for count, img_path in enumerate(class_folder.glob("*.png")):
            with Image.open(img_path).convert('L') as img:
                fh = ImageOps.mirror(img)
                fv = ImageOps.flip(img)
                r90 = img.rotate(90)

                def translate(base_img, dx, dy):
                    return base_img.transform(base_img.size, Image.AFFINE, (1, 0, -dx, 0, 1, -dy), fillcolor=0)

                augs = {
                    "FH": fh, "FV": fv,
                    "R90_FH": ImageOps.mirror(r90), "R90_FV": ImageOps.flip(r90),
                    "T_FH_x_ny": translate(fh, 2, -2), "T_FH_nx_y": translate(fh, -2, 2),
                    "T_FH_nx_ny": translate(fh, -2, -2), "T_FH_x_y": translate(fh, 2, 2),
                    "T_FV_x_ny": translate(fv, 2, -2), "T_FV_nx_y": translate(fv, -2, 2),
                    "T_FV_nx_ny": translate(fv, -2, -2), "T_FV_x_y": translate(fv, 2, 2),
                }

                for name, aug_img in augs.items():
                    target_dir = base_dest / name / class_name
                    target_dir.mkdir(parents=True, exist_ok=True)
                    # MODIFY FILENAME HERE: original.png -> original_FH.png
                    new_filename = f"{img_path.stem}_{name}{img_path.suffix}"
                    aug_img.save(target_dir / new_filename)
        logging.info(f"Processed {count} images in class: {class_name}")

    # 2. Zipping & Sequential Downloads
    logging.info(f"--- Zipping & Downloading {split_name} ---")
    for aug in aug_types:
        aug_path = base_dest / aug
        zip_filename = f"{split_name}_{aug}.zip"

        # Zip only the subfolders (0-9) inside the augmentation folder
        os.system(f"cd {aug_path} && zip -r /content/{zip_filename} . > /dev/null")
        files.download(f"/content/{zip_filename}")

    # 3. Cleanup to free disk space
    logging.info(f"Cleaning up {split_name} temp files...")
    shutil.rmtree(base_dest)

# Execute for Train then Test
process_zip_and_cleanup('/content/test', 'test')

logging.info("All tasks complete. Please ensure your browser allowed multiple file downloads.")

# Merge Test set Augmentations

In [ ]:
import os
from google.colab import files
import zipfile
import shutil
import logging
from pathlib import Path

# --- Setup Logging ---
log_file = "/content/test_set_augmentation.log"
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a' # 'w' overwrites the log each run; use 'a' to append
)

# 1. Setup the final merged directory
merged_test_dir = Path('/content/test_merged')
merged_test_dir.mkdir(exist_ok=True)

# List of the 12 augmentation zips you just created
aug_types = [
    "FH", "FV", "R90_FH", "R90_FV",
    "T_FH_x_ny", "T_FH_nx_y", "T_FH_nx_ny", "T_FH_x_y",
    "T_FV_x_ny", "T_FV_nx_y", "T_FV_nx_ny", "T_FV_x_y"
]
logging.info("--- Starting Merge of Test Augmentations ---")
print(f"Process started. Tracking progress in: {log_file}")

logging.info("--- Starting Merge of Test Augmentations ---")

for aug in aug_types:
    zip_path = f"/content/test_{aug}.zip"
    temp_extract = Path(f"/content/temp_{aug}")

    if os.path.exists(zip_path):
        logging.info(f"Merging {aug}...")

        # Extract zip to a temporary folder
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract)

        # Move files from temp_extract/digit/image.png to merged_test_dir/digit/aug_image.png
        for digit_folder in temp_extract.iterdir():
            if not digit_folder.is_dir(): continue

            target_digit_path = merged_test_dir / digit_folder.name
            target_digit_path.mkdir(exist_ok=True)
            logging.info(f"Processing class: {digit_folder.name}")

            for count, img_file in enumerate(digit_folder.glob("*.png")):
                # Prepend the augmentation name to prevent overwriting
                new_name = f"{aug}_{img_file.name}"
                shutil.move(str(img_file), str(target_digit_path / new_name))
            logging.info(f"Processed {count} images in class: {digit_folder.name}")

        # Clean up the empty temp folder
        shutil.rmtree(temp_extract)
    else:
        print(f"Warning: {zip_path} not found.")

logging.info(f"--- Merge Complete! Total folders in {merged_test_dir}: {len(os.listdir(merged_test_dir))} ---")

# Final Zip of the merged test set
# Using -r for recursive and > /dev/null to keep the output clean
logging.info("Zipping the merged test set... This may take a moment.")
os.system("zip -r test_merged_final.zip /content/test_merged > /dev/null")

print("Triggering download: test_merged_final.zip")
files.download("test_merged_final.zip")

Process started. Tracking progress in: /content/test_set_augmentation.log
--- Starting Merge of Test Augmentations ---
--- Merge Complete! Total folders in /content/test_merged: 10 ---
Zipping the merged test set... This may take a moment.
Triggering download: test_merged_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Merge Train set Augmentations

In [ ]:
import os
from google.colab import files
import zipfile
import shutil
import logging
from pathlib import Path

# --- Setup Logging ---
log_file = "/content/test_set_augmentation.log"
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='w' # 'w' overwrites the log each run; use 'a' to append
)

# 1. Setup the final merged directory
merged_test_dir = Path('/content/train_merged')
merged_test_dir.mkdir(exist_ok=True)

# List of the 12 augmentation zips you just created
aug_types = [
    "FH", "FV", "R90_FH", "R90_FV",
    "T_FH_x_ny", "T_FH_nx_y", "T_FH_nx_ny", "T_FH_x_y",
    "T_FV_x_ny", "T_FV_nx_y", "T_FV_nx_ny", "T_FV_x_y"
]
logging.info("--- Starting Merge of Test Augmentations ---")
print(f"Process started. Tracking progress in: {log_file}")

logging.info("--- Starting Merge of Test Augmentations ---")

for aug in aug_types:
    zip_path = f"/content/{aug}_dataset.zip"
    temp_extract = Path(f"/content/temp_{aug}")

    if os.path.exists(zip_path):
        logging.info(f"Merging {aug}...")

        # Extract zip to a temporary folder
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_extract)

        # Move files from temp_extract/digit/image.png to merged_test_dir/digit/aug_image.png
        for digit_folder in temp_extract.iterdir():
            if not digit_folder.is_dir(): continue

            target_digit_path = merged_test_dir / digit_folder.name
            target_digit_path.mkdir(exist_ok=True)
            logging.info(f"Processing class: {digit_folder.name}")

            for count, img_file in enumerate(digit_folder.glob("*.png")):
                # Prepend the augmentation name to prevent overwriting
                new_name = f"{aug}_{img_file.name}"
                shutil.move(str(img_file), str(target_digit_path / new_name))
            logging.info(f"Processed {count} images in class: {digit_folder.name}")

        # Clean up the empty temp folder
        shutil.rmtree(temp_extract)
    else:
        print(f"Warning: {zip_path} not found.")

logging.info(f"--- Merge Complete! Total folders in {merged_test_dir}: {len(os.listdir(merged_test_dir))} ---")

# Final Zip of the merged test set
# Using -r for recursive and > /dev/null to keep the output clean
logging.info("Zipping the merged train set... This may take a moment.")
os.system("zip -r train_merged_final.zip /content/train_merged > /dev/null")

print("Triggering download: train_merged_final.zip")
files.download("train_merged_final.zip")